# Refinement run — tuned Stage 3 + XGBoost ensemble (Kaggle GPU)

Run this AFTER the full benchmark has finished and you merged `results_and_models.zip`
back into the local repo. Rebuild `etd-repo.zip` (it now contains the trained
channels, processed splits and `tune_and_ensemble.py`) and upload it as a new
version of the `etd-repo` dataset.

This does NOT re-run the benchmark. It reuses the frozen channel nets and:
1. retrains the boosted CNN+BiLSTM with AdamW + cosine LR + longer patience,
2. ensembles it with XGBoost (weight chosen on validation),
3. replaces `residential_model_best.pt` only if the tuned model is better.

Expected runtime on T4: ~20-30 min.

In [ ]:
# Cell 1 — GPU check
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Enable GPU: Settings -> Accelerator -> GPU T4 x2"

In [ ]:
# Cell 2 — copy the dataset (trained channels + processed splits + code) to a writable dir
import shutil, os
shutil.copytree('/kaggle/input/etd-repo', '/kaggle/working/run', dirs_exist_ok=True)
%cd /kaggle/working/run
print(os.listdir('models/checkpoints'))

In [ ]:
# Cell 3 — the refinement run (~20-30 min)
!python -m src.experiments.tune_and_ensemble --config config/config.yaml

In [ ]:
# Cell 4 — package updated weights + results for download
!zip -qr /kaggle/working/results_and_models.zip experiments_results models
print('Download: Output panel -> results_and_models.zip, then unzip -o at the repo root')